# L1_T3 — Model Training: EfficientNet-B0 Multi-Stage Fine-Tuning
### Layer 1 of the Aura pipeline · Task 3 of N

**What this notebook does:** trains the Skin Tone (Fitzpatrick scale) model and a prototype Undertone model,
using the preprocessed archives from `L1_T2` plus one additional public undertone dataset.

**Four-stage transfer learning:**
- **Stage A — FairFace pretraining (proxy task):** EfficientNet-B0 (ImageNet weights) is fine-tuned on
  FairFace's *race* classification task (7 classes). This isn't the real task — it's a bridge: ImageNet was
  trained on general objects, not faces, so adapting the backbone to face/skin-region features first, on a
  large and reasonably diverse dataset (97,698 images), gives Stage B a much better starting point than going
  straight from ImageNet.
- **Stage B — Fitzpatrick17k head training (backbone frozen):** load Stage A's backbone, freeze it, attach a
  new 6-class Fitzpatrick-scale head, and train only that head on the much smaller Fitzpatrick17k set
  (3,734 images).
- **Stage C — Fitzpatrick17k fine-tuning (backbone partially unfrozen):** unfreeze the top layers of the
  backbone and continue training at a low learning rate, so the backbone adapts specifically to skin-tone
  classification rather than staying frozen at "generic face features."
- **Stage D — Undertone head (separate, weak, prototype):** reuse the Stage C backbone (frozen) and train a new
  3-class undertone head (cool / neutral / warm) on a small public Roboflow dataset (980 images).

**Outputs:**
- `skin_tone_classifier.tflite` — Fitzpatrick I–VI (6-class). Intended for app integration, with the accuracy
  caveats in the final summary.
- `undertone_classifier_WEAK.tflite` — Undertone (3-class). **Prototype only, not production-ready.**
  Kept as a separate file so the weak model can never be shipped by accident inside the Fitzpatrick export.

**Important limitation — read before you use the undertone output:** none of the three datasets gathered in
`L1_T1` contain undertone labels. Undertone was always meant to come from the in-house, dermatologist-labeled
Filipino beta dataset, which doesn't exist yet. Stage D uses a small, unverified, single-uploader public dataset
(its own reference model reaches only ~63%), so its output is a weak signal, not a validated one.

**Class imbalance:** Fitzpatrick17k's usable set is imbalanced (161 images for class I vs. 1,082 for class IV).
Stage B/C compensate with per-sample class weights, computed from the training split.

## Step 0 — Mount Drive and load the manifest

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json

DRIVE_ROOT = "/content/drive/MyDrive/Projects/Aura/Layer1_SkinTone"
MANIFEST_PATH = f"{DRIVE_ROOT}/manifest.json"

with open(MANIFEST_PATH) as f:
    manifest = json.load(f)

FAIRFACE_PP_ARCHIVE = manifest["preprocessing"]["fairface"]["drive_archive"]
FITZ_PP_ARCHIVE = manifest["preprocessing"]["fitzpatrick17k"]["drive_archive"]

assert os.path.exists(FAIRFACE_PP_ARCHIVE), f"Not found: {FAIRFACE_PP_ARCHIVE} -- did L1_T2 finish and verify?"
assert os.path.exists(FITZ_PP_ARCHIVE), f"Not found: {FITZ_PP_ARCHIVE} -- did L1_T2 finish and verify?"
print("Both preprocessed archives confirmed present on Drive.")
print(json.dumps(manifest["preprocessing"], indent=2))


Mounted at /content/drive
Both preprocessed archives confirmed present on Drive.
{
  "generated_at_utc": "2026-09-29T05:50:30.342335+00:00",
  "target_image_size": 224,
  "storage_pattern": "Preprocessed outputs stored as single .tar.gz archives per dataset, same pattern as raw data.",
  "fairface": {
    "drive_archive": "/content/drive/MyDrive/Projects/Aura/Layer1_SkinTone/preprocessed/fairface/fairface_preprocessed.tar.gz",
    "archive_size_mb": 1417.1,
    "train_processed": 86744,
    "train_errors": 0,
    "validation_processed": 10954,
    "validation_errors": 0
  },
  "fitzpatrick17k": {
    "drive_archive": "/content/drive/MyDrive/Projects/Aura/Layer1_SkinTone/preprocessed/fitzpatrick17k/fitzpatrick17k_preprocessed.tar.gz",
    "archive_size_mb": 87.6,
    "processed": 3734,
    "errors": 0
  }
}


## Step 0.5 — Local working directories (models/checkpoints use a single-evolving-file pattern)

Checkpoints are NOT saved one-file-per-epoch — that would reintroduce the many-small-files Drive risk from
`L1_T1`. Instead, each stage has exactly two files that get **overwritten in place**: `latest.weights.h5`
(for resuming) and `best.weights.h5` (best validation score so far). Both get copied to Drive and
size-verified after every epoch — two small file copies per epoch is a completely different risk profile than
thousands of new files.

In [ ]:
LOCAL_ROOT = "/content/aura_local/Layer1_SkinTone"
LOCAL_EXTRACTED = f"{LOCAL_ROOT}/extracted_preprocessed"
LOCAL_FAIRFACE_DIR = f"{LOCAL_EXTRACTED}/fairface_preprocessed"
LOCAL_FITZ_DIR = f"{LOCAL_EXTRACTED}/fitzpatrick17k_preprocessed"

LOCAL_CKPT_DIR = f"{LOCAL_ROOT}/checkpoints"
DRIVE_CKPT_DIR = f"{DRIVE_ROOT}/checkpoints"

for d in [LOCAL_EXTRACTED, LOCAL_CKPT_DIR, DRIVE_CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

print("Local root:", LOCAL_ROOT)
print("Drive checkpoint dir:", DRIVE_CKPT_DIR)


Local root: /content/aura_local/Layer1_SkinTone
Drive checkpoint dir: /content/drive/MyDrive/Projects/Aura/Layer1_SkinTone/checkpoints


## Step 1 — Install/confirm dependencies

In [ ]:
!pip install -q scikit-learn pandas tqdm pillow
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


TensorFlow version: 2.20.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Step 2 — Copy + extract the preprocessed archives (copy-first, same fix as L1_T2)

In [ ]:
import shutil, tarfile, time

LOCAL_ARCHIVES = "/content/aura_local/archives"
os.makedirs(LOCAL_ARCHIVES, exist_ok=True)

def copy_and_extract(drive_archive_path, label):
    local_archive = os.path.join(LOCAL_ARCHIVES, os.path.basename(drive_archive_path))
    print(f"Copying {label} from Drive...")
    shutil.copy2(drive_archive_path, local_archive)
    assert os.path.getsize(local_archive) == os.path.getsize(drive_archive_path), f"Copy size mismatch: {label}"
    print(f"  Extracting {label}...")
    with tarfile.open(local_archive, "r:gz") as tar:
        tar.extractall(LOCAL_EXTRACTED, filter="data")
    print(f"  ✅ {label} ready at {LOCAL_EXTRACTED}")

copy_and_extract(FAIRFACE_PP_ARCHIVE, "FairFace (preprocessed)")
copy_and_extract(FITZ_PP_ARCHIVE, "Fitzpatrick17k (preprocessed)")

ff_train_labels = __import__("pandas").read_csv(f"{LOCAL_FAIRFACE_DIR}/train_processed_labels.csv")
ff_val_labels = __import__("pandas").read_csv(f"{LOCAL_FAIRFACE_DIR}/validation_processed_labels.csv")
fitz_labels = __import__("pandas").read_csv(f"{LOCAL_FITZ_DIR}/processed_labels.csv")
print(f"\nFairFace train: {len(ff_train_labels)} | validation: {len(ff_val_labels)}")
print(f"Fitzpatrick17k: {len(fitz_labels)}")


Copying FairFace (preprocessed) from Drive...
  Extracting FairFace (preprocessed)...
  ✅ FairFace (preprocessed) ready at /content/aura_local/Layer1_SkinTone/extracted_preprocessed
Copying Fitzpatrick17k (preprocessed) from Drive...
  Extracting Fitzpatrick17k (preprocessed)...
  ✅ Fitzpatrick17k (preprocessed) ready at /content/aura_local/Layer1_SkinTone/extracted_preprocessed

FairFace train: 86744 | validation: 10954
Fitzpatrick17k: 3734


## Step 3 — A small helper for the "single evolving file, copy + verify" checkpoint pattern

In [ ]:
def save_and_sync_checkpoint(model, local_path, drive_path, label):
    model.save_weights(local_path)
    shutil.copy2(local_path, drive_path)
    local_size = os.path.getsize(local_path)
    drive_size = os.path.getsize(drive_path)
    if local_size != drive_size:
        print(f"  ⚠️  {label}: size mismatch after copy ({local_size} vs {drive_size}) -- will retry next epoch")
    else:
        print(f"  ✅ {label} checkpoint synced to Drive ({drive_size/1e6:.1f} MB)")


## Step 4 — Build the tf.data pipelines

Both pipelines share the same loading/normalization logic. Images are already 224×224 from `L1_T2` — this
just decodes JPEGs and applies EfficientNet's expected preprocessing (not the same as the CLAHE lighting
normalization from `L1_T2`, which corrects photographic lighting; this is a separate, standard step that
puts pixel values into the numeric range EfficientNet's ImageNet-pretrained weights expect).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications.efficientnet import preprocess_input

IMG_SIZE = 224
BATCH_SIZE = 32

def make_dataset(filepaths, labels, num_classes, training, batch_size=BATCH_SIZE):
    labels_onehot = tf.keras.utils.to_categorical(labels, num_classes=num_classes)
    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels_onehot))

    def _load(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.cast(img, tf.float32)
        img = preprocess_input(img)
        return img, label

    ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(2048)
        ds = ds.map(lambda img, lbl: (tf.image.random_flip_left_right(img), lbl),
                    num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


## Step 5 — Stage A: FairFace race-classification pretraining

This trains the *whole* backbone (no freezing) on FairFace's race labels as a proxy task. Race categories are
read directly from the CSV rather than hardcoded, so this doesn't depend on guessing FairFace's exact label
spelling. A modest number of epochs is enough — this stage's job is domain adaptation, not squeezing out
maximum accuracy on a task Aura doesn't actually care about.

In [ ]:
RACE_CLASSES = sorted(ff_train_labels["race"].unique().tolist())
RACE_TO_IDX = {r: i for i, r in enumerate(RACE_CLASSES)}
print("Race classes (from data, not hardcoded):", RACE_CLASSES)

ff_train_labels["race_idx"] = ff_train_labels["race"].map(RACE_TO_IDX)
ff_val_labels["race_idx"] = ff_val_labels["race"].map(RACE_TO_IDX)

ff_train_paths = (f"{LOCAL_FAIRFACE_DIR}/train/" + ff_train_labels["filename"]).tolist()
ff_val_paths = (f"{LOCAL_FAIRFACE_DIR}/validation/" + ff_val_labels["filename"]).tolist()

train_ds_a = make_dataset(ff_train_paths, ff_train_labels["race_idx"].tolist(), len(RACE_CLASSES), training=True)
val_ds_a = make_dataset(ff_val_paths, ff_val_labels["race_idx"].tolist(), len(RACE_CLASSES), training=False)


Race classes (from data, not hardcoded): [0, 1, 2, 3, 4, 5, 6]


In [ ]:
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0

def build_backbone():
    return EfficientNetB0(include_top=False, weights="imagenet", pooling="avg",
                           input_shape=(IMG_SIZE, IMG_SIZE, 3))

backbone_a = build_backbone()
race_head = layers.Dense(len(RACE_CLASSES), activation="softmax", name="race_output")(backbone_a.output)
stage_a_model = Model(inputs=backbone_a.input, outputs=race_head, name="stage_a_fairface_pretrain")

stage_a_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
stage_a_model.summary()


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "stage_a_fairface_pretrain"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 224, 224,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 224, 224,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 224, 224,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 4,058,538 (15.48 MB)

 Trainable params: 4,016,515 (15.32 MB)

 Non-trainable params: 42,023 (164.16 KB)

In [ ]:
STAGE_A_EPOCHS = 3  # domain-adaptation proxy task -- doesn't need to be exhaustively trained

best_val_acc_a = 0.0
LOCAL_CKPT_A_LATEST = f"{LOCAL_CKPT_DIR}/stage_a_latest.weights.h5"
LOCAL_CKPT_A_BEST = f"{LOCAL_CKPT_DIR}/stage_a_best.weights.h5"
DRIVE_CKPT_A_LATEST = f"{DRIVE_CKPT_DIR}/stage_a_latest.weights.h5"
DRIVE_CKPT_A_BEST = f"{DRIVE_CKPT_DIR}/stage_a_best.weights.h5"

for epoch in range(STAGE_A_EPOCHS):
    print(f"\n=== Stage A epoch {epoch+1}/{STAGE_A_EPOCHS} ===")
    history = stage_a_model.fit(train_ds_a, validation_data=val_ds_a, epochs=1, verbose=1)
    val_acc = history.history["val_accuracy"][0]

    save_and_sync_checkpoint(stage_a_model, LOCAL_CKPT_A_LATEST, DRIVE_CKPT_A_LATEST, "Stage A (latest)")
    if val_acc > best_val_acc_a:
        best_val_acc_a = val_acc
        save_and_sync_checkpoint(stage_a_model, LOCAL_CKPT_A_BEST, DRIVE_CKPT_A_BEST, "Stage A (NEW BEST)")

print(f"\nStage A complete. Best validation accuracy on the race proxy task: {best_val_acc_a:.3f}")
print("(This number is not directly meaningful for Aura -- it's a proxy task. What matters is that the")
print(" backbone has now adapted from generic ImageNet features toward face/skin-region features.)")



=== Stage A epoch 1/3 ===
2711/2711 ━━━━━━━━━━━━━━━━━━━━ 527s 159ms/step - accuracy: 0.5528 - loss: 1.1709 - val_accuracy: 0.6410 - val_loss: 0.9482
  ✅ Stage A (latest) checkpoint synced to Drive (49.1 MB)
  ✅ Stage A (NEW BEST) checkpoint synced to Drive (49.1 MB)

=== Stage A epoch 2/3 ===
2711/2711 ━━━━━━━━━━━━━━━━━━━━ 385s 141ms/step - accuracy: 0.6633 - loss: 0.8953 - val_accuracy: 0.6621 - val_loss: 0.8950
  ✅ Stage A (latest) checkpoint synced to Drive (49.1 MB)
  ✅ Stage A (NEW BEST) checkpoint synced to Drive (49.1 MB)

=== Stage A epoch 3/3 ===
2711/2711 ━━━━━━━━━━━━━━━━━━━━ 383s 141ms/step - accuracy: 0.7065 - loss: 0.7810 - val_accuracy: 0.6721 - val_loss: 0.8959
  ✅ Stage A (latest) checkpoint synced to Drive (49.1 MB)
  ✅ Stage A (NEW BEST) checkpoint synced to Drive (49.1 MB)

Stage A complete. Best validation accuracy on the race proxy task: 0.672
(This number is not directly meaningful for Aura -- it's a proxy task. What matters is that the
 backbone has now adapted 

## Step 6 — Stage B: Fitzpatrick17k head training (backbone frozen)

Load Stage A's backbone weights, freeze the backbone entirely, attach a fresh 6-class Fitzpatrick head, and
train only that head. A stratified train/validation split is created here since Fitzpatrick17k doesn't come
with one.

**Implementation notes (Keras 3):**
- Stage A's checkpoint is a full-model `.weights.h5` file (backbone + race head), so it is loaded by rebuilding
  the Stage A architecture around the new backbone. `by_name=True` is not supported for this format.
- Stage B/C use a **single-output model** (Fitzpatrick only). A dual-output model (with a zero-weighted
  undertone head) failed under Keras 3 and was never needed — the undertone head is trained separately in Stage D.
- Class weights are applied as **per-sample weights** inside the dataset, not via `class_weight=`, which is
  unreliable for this setup.

In [ ]:
fitz_labels["fp_idx"] = fitz_labels["fitzpatrick_scale"].astype(int) - 1  # 1..6 -> 0..5

fitz_train_df, fitz_val_df = train_test_split(
    fitz_labels, test_size=0.15, stratify=fitz_labels["fp_idx"], random_state=42
)
print(f"Fitzpatrick17k train: {len(fitz_train_df)} | validation: {len(fitz_val_df)}")

# Class weights to compensate for imbalance (per the original plan's design)
from sklearn.utils.class_weight import compute_class_weight
class_weights_arr = compute_class_weight(
    class_weight="balanced", classes=np.arange(6), y=fitz_train_df["fp_idx"].values
)
class_weights = {i: w for i, w in enumerate(class_weights_arr)}
print("Class weights (higher = rarer class, weighted up):", {k: round(v, 2) for k, v in class_weights.items()})

fitz_train_paths = (f"{LOCAL_FITZ_DIR}/" + fitz_train_df["processed_filename"]).tolist()
fitz_val_paths = (f"{LOCAL_FITZ_DIR}/" + fitz_val_df["processed_filename"]).tolist()

train_ds_b = make_dataset(fitz_train_paths, fitz_train_df["fp_idx"].tolist(), 6, training=True)
val_ds_b = make_dataset(fitz_val_paths, fitz_val_df["fp_idx"].tolist(), 6, training=False)


Fitzpatrick17k train: 3173 | validation: 561
Class weights (higher = rarer class, weighted up): {0: np.float64(3.86), 1: np.float64(0.95), 2: np.float64(0.63), 3: np.float64(0.58), 4: np.float64(0.98), 5: np.float64(3.09)}


In [ ]:
# Fresh backbone, loaded with Stage A's adapted weights (not ImageNet weights directly)
backbone_b = build_backbone()

# Rebuild the exact Stage A architecture around this backbone so the checkpoint structure matches
_tmp_race_head = layers.Dense(len(RACE_CLASSES), activation="softmax", name="race_output")(backbone_b.output)
_tmp_stage_a = Model(inputs=backbone_b.input, outputs=_tmp_race_head)
_tmp_stage_a.load_weights(LOCAL_CKPT_A_BEST)
del _tmp_stage_a, _tmp_race_head              # backbone_b now holds Stage A's weights

backbone_b.trainable = False  # frozen for Stage B

# Single-output model (Fitzpatrick only). The undertone head is intentionally NOT part of this model:
# it was never trained here, and Stage D builds its own undertone model.
fitzpatrick_head = layers.Dense(6, activation="softmax", name="fitzpatrick_output")(backbone_b.output)
stage_b_model = Model(inputs=backbone_b.input, outputs=fitzpatrick_head, name="stage_b_head_training")

stage_b_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
stage_b_model.summary()

Model: "stage_b_head_training"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_8         │ (None, 224, 224,  │          0 │ input_layer_4[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_4     │ (None, 224, 224,  │          7 │ rescaling_8[0][0] │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_9         │ (None, 224, 224,  │          0 │ normalization_4[… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_9[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 4,057,257 (15.48 MB)

 Trainable params: 7,686 (30.02 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [ ]:
# Class weights applied as per-sample weights. Datasets are already batched, so this is batch-aware.
cw_tensor = tf.constant([class_weights[i] for i in range(6)], dtype=tf.float32)

def add_sample_weights(img, fp_label):
    fp_label = tf.cast(fp_label, tf.float32)
    w = tf.reduce_sum(fp_label * cw_tensor, axis=-1)   # one weight per sample, shape (batch,)
    return img, fp_label, w

def cast_labels(img, fp_label):
    return img, tf.cast(fp_label, tf.float32)

train_ds_b2 = train_ds_b.map(add_sample_weights, num_parallel_calls=tf.data.AUTOTUNE)
val_ds_b2 = val_ds_b.map(cast_labels, num_parallel_calls=tf.data.AUTOTUNE)   # validation unweighted

STAGE_B_EPOCHS = 8
best_val_acc_b = 0.0
LOCAL_CKPT_B_LATEST = f"{LOCAL_CKPT_DIR}/stage_b_latest.weights.h5"
LOCAL_CKPT_B_BEST = f"{LOCAL_CKPT_DIR}/stage_b_best.weights.h5"
DRIVE_CKPT_B_LATEST = f"{DRIVE_CKPT_DIR}/stage_b_latest.weights.h5"
DRIVE_CKPT_B_BEST = f"{DRIVE_CKPT_DIR}/stage_b_best.weights.h5"

for epoch in range(STAGE_B_EPOCHS):
    print(f"\n=== Stage B epoch {epoch+1}/{STAGE_B_EPOCHS} ===")
    history = stage_b_model.fit(
        train_ds_b2, validation_data=val_ds_b2, epochs=1, verbose=1
    )
    val_acc = history.history["val_accuracy"][0]

    save_and_sync_checkpoint(stage_b_model, LOCAL_CKPT_B_LATEST, DRIVE_CKPT_B_LATEST, "Stage B (latest)")
    if val_acc > best_val_acc_b:
        best_val_acc_b = val_acc
        save_and_sync_checkpoint(stage_b_model, LOCAL_CKPT_B_BEST, DRIVE_CKPT_B_BEST, "Stage B (NEW BEST)")

print(f"\nStage B complete. Best Fitzpatrick-scale validation accuracy (frozen backbone): {best_val_acc_b:.3f}")


=== Stage B epoch 1/8 ===
100/100 ━━━━━━━━━━━━━━━━━━━━ 95s 688ms/step - accuracy: 0.2414 - loss: 1.6090 - val_accuracy: 0.3030 - val_loss: 1.5415
  ✅ Stage B (latest) checkpoint synced to Drive (16.9 MB)
  ✅ Stage B (NEW BEST) checkpoint synced to Drive (16.9 MB)

=== Stage B epoch 2/8 ===
100/100 ━━━━━━━━━━━━━━━━━━━━ 7s 47ms/step - accuracy: 0.3476 - loss: 1.4044 - val_accuracy: 0.3102 - val_loss: 1.5683
  ✅ Stage B (latest) checkpoint synced to Drive (16.9 MB)
  ✅ Stage B (NEW BEST) checkpoint synced to Drive (16.9 MB)

=== Stage B epoch 3/8 ===
100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 41ms/step - accuracy: 0.3587 - loss: 1.3403 - val_accuracy: 0.3494 - val_loss: 1.4735
  ✅ Stage B (latest) checkpoint synced to Drive (16.9 MB)
  ✅ Stage B (NEW BEST) checkpoint synced to Drive (16.9 MB)

=== Stage B epoch 4/8 ===
100/100 ━━━━━━━━━━━━━━━━━━━━ 6s 46ms/step - accuracy: 0.3946 - loss: 1.2745 - val_accuracy: 0.3601 - val_loss: 1.4664
  ✅ Stage B (latest) checkpoint synced to Drive (16.9 MB)
  ✅ St

**Note on class weights:** they are applied as per-sample weights in the training dataset (each sample's weight
is looked up from its one-hot Fitzpatrick label). Validation data is left unweighted so the reported validation
accuracy is not inflated by rare-class weighting. Stage C reuses the same weighted dataset.

## Step 7 — Stage C: Unfreeze and fine-tune end-to-end

Unfreeze the top portion of the backbone and continue training everything together at a much lower learning
rate — standard fine-tuning practice: a frozen-backbone warm start (Stage B) avoids destroying the pretrained
features with large early gradients, then a low-LR full fine-tune (Stage C) lets the backbone specialize.

In [ ]:
# Load Stage B's best weights, then unfreeze the top ~20% of backbone layers
stage_b_model.load_weights(LOCAL_CKPT_B_BEST)

backbone_layers = [l for l in stage_b_model.layers if l.name.startswith("block")]
unfreeze_from = int(len(backbone_layers) * 0.8)  # unfreeze the last 20% of blocks
for layer in backbone_layers[:unfreeze_from]:
    layer.trainable = False
for layer in backbone_layers[unfreeze_from:]:
    layer.trainable = True

stage_b_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # much lower LR for fine-tuning
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
print(f"Unfroze {len(backbone_layers) - unfreeze_from} of {len(backbone_layers)} backbone blocks for fine-tuning.")

Unfroze 46 of 227 backbone blocks for fine-tuning.


In [ ]:
STAGE_C_EPOCHS = 10
best_val_acc_c = 0.0
LOCAL_CKPT_C_LATEST = f"{LOCAL_CKPT_DIR}/stage_c_latest.weights.h5"
LOCAL_CKPT_C_BEST = f"{LOCAL_CKPT_DIR}/stage_c_best.weights.h5"
DRIVE_CKPT_C_LATEST = f"{DRIVE_CKPT_DIR}/stage_c_latest.weights.h5"
DRIVE_CKPT_C_BEST = f"{DRIVE_CKPT_DIR}/stage_c_best.weights.h5"

for epoch in range(STAGE_C_EPOCHS):
    print(f"\n=== Stage C epoch {epoch+1}/{STAGE_C_EPOCHS} ===")
    history = stage_b_model.fit(
        train_ds_b2, validation_data=val_ds_b2, epochs=1, verbose=1
    )
    val_acc = history.history["val_accuracy"][0]

    save_and_sync_checkpoint(stage_b_model, LOCAL_CKPT_C_LATEST, DRIVE_CKPT_C_LATEST, "Stage C (latest)")
    if val_acc > best_val_acc_c:
        best_val_acc_c = val_acc
        save_and_sync_checkpoint(stage_b_model, LOCAL_CKPT_C_BEST, DRIVE_CKPT_C_BEST, "Stage C (NEW BEST)")

print(f"\nStage C complete. Best Fitzpatrick-scale validation accuracy (fine-tuned): {best_val_acc_c:.3f}")
print(f"(Compare to Stage B's {best_val_acc_b:.3f} frozen-backbone result -- Stage C should be higher.)")


=== Stage C epoch 1/10 ===
100/100 ━━━━━━━━━━━━━━━━━━━━ 64s 320ms/step - accuracy: 0.3813 - loss: 1.3044 - val_accuracy: 0.3547 - val_loss: 1.4710
  ✅ Stage C (latest) checkpoint synced to Drive (32.1 MB)
  ✅ Stage C (NEW BEST) checkpoint synced to Drive (32.1 MB)

=== Stage C epoch 2/10 ===
100/100 ━━━━━━━━━━━━━━━━━━━━ 7s 58ms/step - accuracy: 0.3876 - loss: 1.2797 - val_accuracy: 0.3529 - val_loss: 1.4805
  ✅ Stage C (latest) checkpoint synced to Drive (32.1 MB)

=== Stage C epoch 3/10 ===
100/100 ━━━━━━━━━━━━━━━━━━━━ 7s 52ms/step - accuracy: 0.4201 - loss: 1.2469 - val_accuracy: 0.3529 - val_loss: 1.4832
  ✅ Stage C (latest) checkpoint synced to Drive (32.1 MB)

=== Stage C epoch 4/10 ===
100/100 ━━━━━━━━━━━━━━━━━━━━ 8s 57ms/step - accuracy: 0.4239 - loss: 1.2301 - val_accuracy: 0.3529 - val_loss: 1.4776
  ✅ Stage C (latest) checkpoint synced to Drive (32.1 MB)

=== Stage C epoch 5/10 ===
100/100 ━━━━━━━━━━━━━━━━━━━━ 6s 52ms/step - accuracy: 0.4378 - loss: 1.2101 - val_accuracy: 0.

## Step 8 — Evaluate: overall AND per-class accuracy

Overall accuracy alone can hide a model that's great on common classes (II, III) and bad on the ones Aura
actually cares about (IV–VI). Check both.

In [ ]:
stage_b_model.load_weights(LOCAL_CKPT_C_BEST)

y_true, y_pred = [], []
for imgs, labels in val_ds_b2:
    preds = stage_b_model.predict(imgs, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1).tolist())
    y_true.extend(np.argmax(labels.numpy(), axis=1).tolist())

from sklearn.metrics import classification_report, confusion_matrix

print("Per-class report (class 0 = Fitzpatrick I, class 5 = Fitzpatrick VI):")
print(classification_report(y_true, y_pred, target_names=[f"Fitzpatrick {i+1}" for i in range(6)]))

print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))

Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 88 variables. 


Per-class report (class 0 = Fitzpatrick I, class 5 = Fitzpatrick VI):
               precision    recall  f1-score   support

Fitzpatrick 1       0.20      0.54      0.30        24
Fitzpatrick 2       0.35      0.40      0.38        99
Fitzpatrick 3       0.45      0.35      0.39       150
Fitzpatrick 4       0.44      0.26      0.33       163
Fitzpatrick 5       0.36      0.39      0.37        95
Fitzpatrick 6       0.32      0.67      0.43        30

     accuracy                           0.37       561
    macro avg       0.35      0.44      0.37       561
 weighted avg       0.40      0.37      0.37       561

Confusion matrix:
[[13  9  0  1  1  0]
 [23 40 23  5  3  5]
 [17 38 53 28 13  1]
 [ 6 24 35 43 44 11]
 [ 3  3  8 18 37 26]
 [ 2  0  0  2  6 20]]


In [ ]:
import numpy as np
y_t, y_p = np.array(y_true), np.array(y_pred)
print("Exact accuracy: ", round((y_t == y_p).mean(), 3))
print("Within ±1 class:", round((np.abs(y_t - y_p) <= 1).mean(), 3))

Exact accuracy:  0.367
Within ±1 class: 0.813


## Step 8.5 — Stage D: Undertone head training (real data, with honest limits)

Fitzpatrick17k and FairFace have no undertone labels — there's nothing to add to Stages A/B/C. Instead, this
uses a small, separately-sourced dataset to give the undertone head *some* real training instead of leaving
it at random initialization.

**What's being used, and its limits — read this before trusting the output:**
- **Roboflow "Undertone Classification Dataset"** (CC BY 4.0, public): 408–980 images, labeled neutral/cool/warm.
  Single anonymous uploader, no published labeling methodology, no stated ethnic/geographic breakdown, and
  their own reference model only reaches 63% accuracy on this 3-class task. This is a weak-but-real training
  signal, not a validated one.
- **A rule-based LAB-colorspace classifier** (no training data needed at all) is added as a second, independent
  method — useful specifically because it doesn't inherit whatever label noise or bias sits in the Roboflow set.
- **Neither replaces the in-house, dermatologist-labeled Filipino dataset** your plan already calls for. This
  stage exists so the undertone head is "weakly informed" instead of "pure noise" in the meantime — not so
  it's production-ready.

In [ ]:
# ---- RECOVERY CELL: restores notebook state after a runtime restart (no retraining) ----
from google.colab import drive
drive.mount('/content/drive')

import os, json, shutil, time, glob, tarfile
import numpy as np
import pandas as pd
import tensorflow as tf
import cv2
from PIL import Image
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input

DRIVE_ROOT = "/content/drive/MyDrive/Projects/Aura/Layer1_SkinTone"
MANIFEST_PATH = f"{DRIVE_ROOT}/manifest.json"
with open(MANIFEST_PATH) as f:
    manifest = json.load(f)

LOCAL_ROOT = "/content/aura_local/Layer1_SkinTone"
LOCAL_EXTRACTED = f"{LOCAL_ROOT}/extracted_preprocessed"
LOCAL_FITZ_DIR = f"{LOCAL_EXTRACTED}/fitzpatrick17k_preprocessed"
LOCAL_CKPT_DIR = f"{LOCAL_ROOT}/checkpoints"
DRIVE_CKPT_DIR = f"{DRIVE_ROOT}/checkpoints"
for d in [LOCAL_EXTRACTED, LOCAL_CKPT_DIR, DRIVE_CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 32

def save_and_sync_checkpoint(model, local_path, drive_path, label):
    model.save_weights(local_path)
    shutil.copy2(local_path, drive_path)
    local_size, drive_size = os.path.getsize(local_path), os.path.getsize(drive_path)
    if local_size != drive_size:
        print(f"  ⚠️  {label}: size mismatch after copy ({local_size} vs {drive_size})")
    else:
        print(f"  ✅ {label} checkpoint synced to Drive ({drive_size/1e6:.1f} MB)")

def make_dataset(filepaths, labels, num_classes, training, batch_size=BATCH_SIZE):
    labels_onehot = tf.keras.utils.to_categorical(labels, num_classes=num_classes)
    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels_onehot))
    def _load(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.cast(img, tf.float32)
        img = preprocess_input(img)
        return img, label
    ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(2048)
        ds = ds.map(lambda img, lbl: (tf.image.random_flip_left_right(img), lbl),
                    num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

def build_backbone():
    return EfficientNetB0(include_top=False, weights="imagenet", pooling="avg",
                          input_shape=(IMG_SIZE, IMG_SIZE, 3))

# --- Fitzpatrick data: re-extract only if the local disk was wiped ---
if not os.path.isdir(LOCAL_FITZ_DIR):
    print("Local Fitzpatrick data missing -- re-extracting from Drive...")
    FITZ_PP_ARCHIVE = manifest["preprocessing"]["fitzpatrick17k"]["drive_archive"]
    LOCAL_ARCHIVES = "/content/aura_local/archives"
    os.makedirs(LOCAL_ARCHIVES, exist_ok=True)
    local_archive = os.path.join(LOCAL_ARCHIVES, os.path.basename(FITZ_PP_ARCHIVE))
    shutil.copy2(FITZ_PP_ARCHIVE, local_archive)
    with tarfile.open(local_archive, "r:gz") as tar:
        tar.extractall(LOCAL_EXTRACTED, filter="data")

fitz_labels = pd.read_csv(f"{LOCAL_FITZ_DIR}/processed_labels.csv")
fitz_labels["fp_idx"] = fitz_labels["fitzpatrick_scale"].astype(int) - 1
fitz_train_df, fitz_val_df = train_test_split(
    fitz_labels, test_size=0.15, stratify=fitz_labels["fp_idx"], random_state=42   # same split as before
)
fitz_val_paths = (f"{LOCAL_FITZ_DIR}/" + fitz_val_df["processed_filename"]).tolist()
val_ds_b = make_dataset(fitz_val_paths, fitz_val_df["fp_idx"].tolist(), 6, training=False)
val_ds_b2 = val_ds_b.map(lambda img, lbl: (img, tf.cast(lbl, tf.float32)), num_parallel_calls=tf.data.AUTOTUNE)

# --- Checkpoint paths (pull from Drive if the local copy is gone) ---
LOCAL_CKPT_C_BEST = f"{LOCAL_CKPT_DIR}/stage_c_best.weights.h5"
DRIVE_CKPT_C_BEST = f"{DRIVE_CKPT_DIR}/stage_c_best.weights.h5"
if not os.path.exists(LOCAL_CKPT_C_BEST):
    shutil.copy2(DRIVE_CKPT_C_BEST, LOCAL_CKPT_C_BEST)

# --- Rebuild the Stage B/C model and load Stage C's best weights ---
backbone_b = build_backbone()
fitzpatrick_head = layers.Dense(6, activation="softmax", name="fitzpatrick_output")(backbone_b.output)
stage_b_model = Model(inputs=backbone_b.input, outputs=fitzpatrick_head, name="stage_b_head_training")
stage_b_model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                      loss="categorical_crossentropy", metrics=["accuracy"])
stage_b_model.load_weights(LOCAL_CKPT_C_BEST)
backbone_b.trainable = False   # backbone_b now holds Stage C's fine-tuned weights

DRIVE_CKPT_A_BEST = f"{DRIVE_CKPT_DIR}/stage_a_best.weights.h5"
DRIVE_CKPT_B_BEST = f"{DRIVE_CKPT_DIR}/stage_b_best.weights.h5"
DRIVE_CKPT_C_BEST = f"{DRIVE_CKPT_DIR}/stage_c_best.weights.h5"

# Values from your earlier logs (needed later by the manifest cell):
best_val_acc_b = 0.360
best_val_acc_c = 0.367
best_val_acc_a = 0.0   # <-- replace with your final Stage A val accuracy from the log, if you still have it

print("✅ State restored. Stage C weights loaded into backbone_b.")

Mounted at /content/drive
Local Fitzpatrick data missing -- re-extracting from Drive...
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
✅ State restored. Stage C weights loaded into backbone_b.


Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 88 variables. 


In [ ]:
!pip install -q roboflow

import roboflow
from google.colab import userdata

ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")   # stored in Colab Secrets, not in the notebook

rf = roboflow.Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("undertone").project("undertone-classification")
version = project.version(1)

LOCAL_UNDERTONE_DIR = "/content/aura_local/undertone_dataset"
dataset = version.download(model_format="folder", location=LOCAL_UNDERTONE_DIR)
print("Downloaded to:", LOCAL_UNDERTONE_DIR)

import subprocess
print(subprocess.run(["find", LOCAL_UNDERTONE_DIR, "-maxdepth", "2", "-type", "d"], capture_output=True, text=True).stdout)

loading Roboflow workspace...
loading Roboflow project...
Downloaded to: /content/aura_local/undertone_dataset
/content/aura_local/undertone_dataset
/content/aura_local/undertone_dataset/test
/content/aura_local/undertone_dataset/test/neutral
/content/aura_local/undertone_dataset/test/warm
/content/aura_local/undertone_dataset/test/cool
/content/aura_local/undertone_dataset/valid
/content/aura_local/undertone_dataset/valid/neutral
/content/aura_local/undertone_dataset/valid/warm
/content/aura_local/undertone_dataset/valid/cool
/content/aura_local/undertone_dataset/train
/content/aura_local/undertone_dataset/train/neutral
/content/aura_local/undertone_dataset/train/warm
/content/aura_local/undertone_dataset/train/cool



In [ ]:
import glob
import cv2
from PIL import Image

def collect_folder_dataset(root_dir):
    """Handles Roboflow's typical train/valid/test split-by-subfolder-per-class layout."""
    records = []
    for split_dir in ["train", "valid", "test"]:
        split_path = os.path.join(root_dir, split_dir)
        if not os.path.isdir(split_path):
            continue
        for class_name in os.listdir(split_path):
            class_path = os.path.join(split_path, class_name)
            if not os.path.isdir(class_path):
                continue
            for img_path in glob.glob(os.path.join(class_path, "*.jpg")) + glob.glob(os.path.join(class_path, "*.png")):
                records.append({"filepath": img_path, "class_name": class_name.lower()})
    return pd.DataFrame(records)

undertone_df = collect_folder_dataset(LOCAL_UNDERTONE_DIR)
print(f"Total images found: {len(undertone_df)}")
print(undertone_df["class_name"].value_counts())

UNDERTONE_CLASSES = sorted(undertone_df["class_name"].unique().tolist())
UNDERTONE_TO_IDX = {c: i for i, c in enumerate(UNDERTONE_CLASSES)}
print("\nClass order (index 0, 1, 2):", UNDERTONE_CLASSES)
undertone_df["undertone_idx"] = undertone_df["class_name"].map(UNDERTONE_TO_IDX)

Total images found: 980
class_name
neutral    429
warm       413
cool       138
Name: count, dtype: int64

Class order (index 0, 1, 2): ['cool', 'neutral', 'warm']


In [ ]:
def classify_undertone_rule_based(img_rgb):
    """
    HEURISTIC, NOT CALIBRATED. Classifies undertone via hue angle in LAB's a*-b* plane:
    warm skin skews toward yellow (higher b* relative to a*), cool skews toward pink/blue
    (higher a* relative to b*). Thresholds below are a starting approximation -- validate
    against real examples with known ground truth before trusting this in production.
    """
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    l, a, b = cv2.split(lab)
    a_signed, b_signed = a - 128.0, b - 128.0

    mask = (l > 50) & (l < 200)  # exclude deep shadow / blown-out highlight pixels
    if mask.sum() < 100:
        mask = np.ones_like(l, dtype=bool)

    a_mean, b_mean = float(a_signed[mask].mean()), float(b_signed[mask].mean())
    chroma = (a_mean**2 + b_mean**2) ** 0.5
    hue_angle = float(np.degrees(np.arctan2(b_mean, a_mean)) % 360)

    if chroma < 3:  # near-gray; hue angle is unstable/meaningless at very low chroma
        return "neutral", hue_angle, chroma
    elif hue_angle <= 55:
        return "warm", hue_angle, chroma
    elif hue_angle <= 85:
        return "neutral", hue_angle, chroma
    else:
        return "cool", hue_angle, chroma

# Quick smoke test on a few Roboflow images with known labels
sample = undertone_df.sample(min(5, len(undertone_df)), random_state=42)
for _, row in sample.iterrows():
    img = np.array(Image.open(row["filepath"]).convert("RGB").resize((224, 224)))
    pred, angle, chroma = classify_undertone_rule_based(img)
    print(f"True: {row['class_name']:8s} | Rule-based: {pred:8s} | hue={angle:.1f} chroma={chroma:.1f}")

True: cool     | Rule-based: neutral  | hue=56.8 chroma=20.9
True: warm     | Rule-based: warm     | hue=51.0 chroma=30.6
True: neutral  | Rule-based: neutral  | hue=69.7 chroma=8.8
True: warm     | Rule-based: warm     | hue=37.1 chroma=18.6
True: neutral  | Rule-based: neutral  | hue=58.2 chroma=42.5


In [ ]:
def make_dataset_resized(filepaths, labels, num_classes, training, batch_size=16):
    """Like make_dataset, but resizes to IMG_SIZE (Roboflow images aren't pre-resized)."""
    labels_onehot = tf.keras.utils.to_categorical(labels, num_classes=num_classes)
    ds = tf.data.Dataset.from_tensor_slices((filepaths, labels_onehot))

    def _load(path, label):
        img = tf.io.read_file(path)
        img = tf.io.decode_image(img, channels=3, expand_animations=False)
        img.set_shape([None, None, 3])
        img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
        img = tf.cast(img, tf.float32)
        img = preprocess_input(img)
        return img, label

    ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(1024)
        ds = ds.map(lambda img, lbl: (tf.image.random_flip_left_right(img), lbl),
                    num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

undertone_train_df, undertone_val_df = train_test_split(
    undertone_df, test_size=0.2, stratify=undertone_df["undertone_idx"], random_state=42
)
print(f"Undertone train: {len(undertone_train_df)} | validation: {len(undertone_val_df)}")

train_ds_d = make_dataset_resized(undertone_train_df["filepath"].tolist(), undertone_train_df["undertone_idx"].tolist(),
                                  num_classes=3, training=True)
val_ds_d = make_dataset_resized(undertone_val_df["filepath"].tolist(), undertone_val_df["undertone_idx"].tolist(),
                                num_classes=3, training=False)

# Reuse Stage C's fully fine-tuned backbone (still in memory as `backbone_b`), frozen.
# (It holds the best Stage C weights because the evaluation cell loaded LOCAL_CKPT_C_BEST into stage_b_model,
#  which shares these layers. If your Colab session restarted, reload those weights first.)
backbone_b.trainable = False
new_undertone_head = layers.Dense(3, activation="softmax", name="undertone_output_v2")(backbone_b.output)
stage_d_model = Model(inputs=backbone_b.input, outputs=new_undertone_head, name="stage_d_undertone")

stage_d_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

STAGE_D_EPOCHS = 10
best_val_acc_d = 0.0
LOCAL_CKPT_D_BEST = f"{LOCAL_CKPT_DIR}/stage_d_best.weights.h5"
DRIVE_CKPT_D_BEST = f"{DRIVE_CKPT_DIR}/stage_d_best.weights.h5"

for epoch in range(STAGE_D_EPOCHS):
    print(f"\n=== Stage D epoch {epoch+1}/{STAGE_D_EPOCHS} ===")
    history = stage_d_model.fit(train_ds_d, validation_data=val_ds_d, epochs=1, verbose=1)
    val_acc = history.history["val_accuracy"][0]
    if val_acc > best_val_acc_d:
        best_val_acc_d = val_acc
        save_and_sync_checkpoint(stage_d_model, LOCAL_CKPT_D_BEST, DRIVE_CKPT_D_BEST, "Stage D (NEW BEST)")

print(f"\nStage D complete. Best undertone validation accuracy: {best_val_acc_d:.3f}")
print("(Roboflow's own reference model reports 63% on this dataset -- use that as your sanity-check ceiling,")
print(" not 100%. This is a small, unverified dataset -- treat this number accordingly.)")

Undertone train: 784 | validation: 196

=== Stage D epoch 1/10 ===
49/49 ━━━━━━━━━━━━━━━━━━━━ 51s 424ms/step - accuracy: 0.4107 - loss: 1.0536 - val_accuracy: 0.5357 - val_loss: 0.9672
  ✅ Stage D (NEW BEST) checkpoint synced to Drive (16.9 MB)

=== Stage D epoch 2/10 ===
49/49 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - accuracy: 0.5115 - loss: 0.9618 - val_accuracy: 0.5306 - val_loss: 0.9515

=== Stage D epoch 3/10 ===
49/49 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.5548 - loss: 0.9108 - val_accuracy: 0.5204 - val_loss: 0.9526

=== Stage D epoch 4/10 ===
49/49 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.6071 - loss: 0.8548 - val_accuracy: 0.5663 - val_loss: 0.9435
  ✅ Stage D (NEW BEST) checkpoint synced to Drive (16.9 MB)

=== Stage D epoch 5/10 ===
49/49 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.6071 - loss: 0.8322 - val_accuracy: 0.5561 - val_loss: 0.9364

=== Stage D epoch 6/10 ===
49/49 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - accuracy: 0.6531 - loss: 0.7981 - val_accuracy: 0

In [ ]:
stage_d_model.load_weights(LOCAL_CKPT_D_BEST)

y_true_u, y_pred_learned, y_pred_rulebased = [], [], []
for _, row in undertone_val_df.iterrows():
    img = np.array(Image.open(row["filepath"]).convert("RGB").resize((224, 224)))
    img_batch = preprocess_input(img.astype(np.float32))[np.newaxis, ...]

    learned_pred = np.argmax(stage_d_model.predict(img_batch, verbose=0)[0])
    rule_pred_name, _, _ = classify_undertone_rule_based(img)

    y_true_u.append(row["undertone_idx"])
    y_pred_learned.append(learned_pred)
    y_pred_rulebased.append(UNDERTONE_TO_IDX.get(rule_pred_name, -1))

from sklearn.metrics import accuracy_score
print("Learned head accuracy:   ", round(accuracy_score(y_true_u, y_pred_learned), 3))
print("Rule-based accuracy:     ", round(accuracy_score(y_true_u, y_pred_rulebased), 3))
agreement = sum(a == b for a, b in zip(y_pred_learned, y_pred_rulebased)) / len(y_pred_learned)
print("Agreement between the two methods:", round(agreement, 3))

Learned head accuracy:    0.592
Rule-based accuracy:      0.357
Agreement between the two methods: 0.383


## Step 9 — Export to TFLite (two separate models)

Two TFLite files are exported, deliberately kept separate:

1. **`undertone_classifier_WEAK.tflite`** (Stage D head): 3-class undertone softmax. It is trained on a small,
   unverified public dataset (val accuracy ~0.60), so it is flagged `_WEAK` in the filename and the manifest.
   Do not show its output to end users until it has been validated on real, dermatologist-labeled data.
2. **`skin_tone_classifier.tflite`** (Stage C model): 6-class Fitzpatrick softmax (I–VI). This is the model
   intended for app integration.

Keeping them as separate files means the weak undertone model can never ship by accident inside the Fitzpatrick
export. Both take a 224×224×3 float32 image (raw RGB, 0–255; EfficientNet rescaling is built in).

In [ ]:
# Undertone export -- kept SEPARATE from the Fitzpatrick model, clearly flagged as weak/unverified
undertone_export_model = Model(inputs=stage_d_model.input, outputs=stage_d_model.output,
                                name="undertone_classifier_export_WEAK")
converter_u = tf.lite.TFLiteConverter.from_keras_model(undertone_export_model)
converter_u.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_undertone = converter_u.convert()

LOCAL_TFLITE_UNDERTONE_PATH = f"{LOCAL_CKPT_DIR}/undertone_classifier_WEAK.tflite"
with open(LOCAL_TFLITE_UNDERTONE_PATH, "wb") as f:
    f.write(tflite_undertone)
print(f"Undertone TFLite exported (WEAK, {os.path.getsize(LOCAL_TFLITE_UNDERTONE_PATH)/1e6:.2f} MB) -- "
      f"trained on {len(undertone_df)} unverified images, val accuracy {best_val_acc_d:.3f}. "
      f"Do not treat this as production-ready.")

DRIVE_TFLITE_UNDERTONE_PATH = f"{DRIVE_CKPT_DIR}/undertone_classifier_WEAK.tflite"
shutil.copy2(LOCAL_TFLITE_UNDERTONE_PATH, DRIVE_TFLITE_UNDERTONE_PATH)
assert os.path.getsize(LOCAL_TFLITE_UNDERTONE_PATH) == os.path.getsize(DRIVE_TFLITE_UNDERTONE_PATH)
print("✅ Undertone TFLite synced to Drive (kept separate from the Fitzpatrick export).")

Saved artifact at '/tmp/tmp36ekpxhb'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  137912666501392: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  137911370827152: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  137913932494992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137913932493840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137913932492880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137913932496144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137913932495376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137913932492688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137913823264848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137913823265232: TensorSpec(shape=(), dtype=tf.resource, name=None

In [ ]:
stage_b_model.load_weights(LOCAL_CKPT_C_BEST)   # make sure the export uses Stage C's best weights

export_model = Model(inputs=stage_b_model.input,
                     outputs=stage_b_model.get_layer("fitzpatrick_output").output,
                     name="skin_tone_classifier_export")

converter = tf.lite.TFLiteConverter.from_keras_model(export_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # post-training quantization (float16-ish dynamic range)
tflite_model = converter.convert()

LOCAL_TFLITE_PATH = f"{LOCAL_CKPT_DIR}/skin_tone_classifier.tflite"
with open(LOCAL_TFLITE_PATH, "wb") as f:
    f.write(tflite_model)

tflite_size_mb = os.path.getsize(LOCAL_TFLITE_PATH) / 1e6
print(f"TFLite model exported: {LOCAL_TFLITE_PATH} ({tflite_size_mb:.2f} MB)")
print("Output: single 6-class softmax (Fitzpatrick I-VI). Undertone is NOT included in this export.")

Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 88 variables. 


Saved artifact at '/tmp/tmp44vdbtzk'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 6), dtype=tf.float32, name=None)
Captures:
  137910669491856: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  137910669491472: TensorSpec(shape=(1, 1, 1, 3), dtype=tf.float32, name=None)
  137913932494992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137913932493840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137913932492880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137913932496144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137913932495376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137913932492688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137913823264848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137913823265232: TensorSpec(shape=(), dtype=tf.resource, name=None

In [ ]:
# Sanity-check the exported TFLite model actually runs and gives a 6-class output
interpreter = tf.lite.Interpreter(model_path=LOCAL_TFLITE_PATH)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input shape:", input_details[0]["shape"])
print("Output shape:", output_details[0]["shape"])

sample_img, _ = next(iter(val_ds_b2))
interpreter.set_tensor(input_details[0]["index"], sample_img[:1].numpy())
interpreter.invoke()
output = interpreter.get_tensor(output_details[0]["index"])
print("Sample output (should be 6 probabilities summing to ~1.0):", output)
print("Sum check:", output.sum())


Input shape: [  1 224 224   3]
Output shape: [1 6]
Sample output (should be 6 probabilities summing to ~1.0): [[0.06734839 0.06115364 0.07423747 0.06554659 0.27925524 0.45245865]]
Sum check: 1.0


    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    


In [ ]:
DRIVE_TFLITE_PATH = f"{DRIVE_CKPT_DIR}/skin_tone_classifier.tflite"
shutil.copy2(LOCAL_TFLITE_PATH, DRIVE_TFLITE_PATH)
assert os.path.getsize(LOCAL_TFLITE_PATH) == os.path.getsize(DRIVE_TFLITE_PATH), "TFLite copy size mismatch"
print(f"✅ TFLite model synced to Drive: {DRIVE_TFLITE_PATH}")


✅ TFLite model synced to Drive: /content/drive/MyDrive/Projects/Aura/Layer1_SkinTone/checkpoints/skin_tone_classifier.tflite


## Step 10 — Force a full Drive sync and re-verify, then update the manifest

In [ ]:
drive.flush_and_unmount()
time.sleep(3)
drive.mount('/content/drive')
print("Drive remounted -- re-verifying from a fresh mount.")

check_paths = {
    "Stage A best checkpoint": DRIVE_CKPT_A_BEST,
    "Stage B best checkpoint": DRIVE_CKPT_B_BEST,
    "Stage C best checkpoint": DRIVE_CKPT_C_BEST,
    "Stage D best checkpoint (undertone)": DRIVE_CKPT_D_BEST,
    "TFLite export (Fitzpatrick)": DRIVE_TFLITE_PATH,
    "TFLite export (undertone, WEAK)": DRIVE_TFLITE_UNDERTONE_PATH,
}
all_ok = True
for label, path in check_paths.items():
    if os.path.exists(path):
        print(f"✅ {label}: {os.path.getsize(path)/1e6:.2f} MB")
    else:
        print(f"❌ MISSING: {label} at {path}")
        all_ok = False

print("\n✅✅ All good." if all_ok else "\n⚠️ Something is missing -- do not close this session yet.")

Mounted at /content/drive
Drive remounted -- re-verifying from a fresh mount.
✅ Stage A best checkpoint: 49.14 MB
✅ Stage B best checkpoint: 16.91 MB
✅ Stage C best checkpoint: 32.09 MB
✅ Stage D best checkpoint (undertone): 16.86 MB
✅ TFLite export (Fitzpatrick): 4.55 MB
✅ TFLite export (undertone, WEAK): 4.55 MB

✅✅ All good.


In [ ]:
from datetime import datetime, timezone

# y_true / y_pred come from the Step 8 evaluation cell -- run that first
within_1 = float((np.abs(np.array(y_true) - np.array(y_pred)) <= 1).mean())

manifest["training"] = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "stage_a_race_proxy_val_accuracy": round(float(best_val_acc_a), 4),
    "stage_b_frozen_backbone_val_accuracy": round(float(best_val_acc_b), 4),
    "stage_c_finetuned_val_accuracy": round(float(best_val_acc_c), 4),
    "stage_c_within_1_class_accuracy": round(within_1, 4),
    "stage_d_undertone_val_accuracy": round(float(best_val_acc_d), 4),
    "tflite_export": {
        "drive_path": DRIVE_TFLITE_PATH,
        "size_mb": round(tflite_size_mb, 2),
        "output": "Fitzpatrick scale only (6-class softmax).",
    },
    "tflite_undertone_export_WEAK": {
        "drive_path": DRIVE_TFLITE_UNDERTONE_PATH,
        "output": "3-class undertone softmax. WEAK: trained on a small unverified public dataset. Not production-ready.",
    },
    "checkpoints_drive_dir": DRIVE_CKPT_DIR,
}

with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2)

print("Manifest updated:", MANIFEST_PATH)
print(json.dumps(manifest["training"], indent=2))

Manifest updated: /content/drive/MyDrive/Projects/Aura/Layer1_SkinTone/manifest.json
{
  "generated_at_utc": "2026-09-30T16:16:41.830488+00:00",
  "stage_a_race_proxy_val_accuracy": 0.0,
  "stage_b_frozen_backbone_val_accuracy": 0.36,
  "stage_c_finetuned_val_accuracy": 0.367,
  "stage_c_within_1_class_accuracy": 0.8128,
  "stage_d_undertone_val_accuracy": 0.602,
  "tflite_export": {
    "drive_path": "/content/drive/MyDrive/Projects/Aura/Layer1_SkinTone/checkpoints/skin_tone_classifier.tflite",
    "size_mb": 4.55,
    "output": "Fitzpatrick scale only (6-class softmax)."
  },
  "tflite_undertone_export_WEAK": {
    "drive_path": "/content/drive/MyDrive/Projects/Aura/Layer1_SkinTone/checkpoints/undertone_classifier_WEAK.tflite",
    "output": "3-class undertone softmax. WEAK: trained on a small unverified public dataset. Not production-ready."
  },
  "checkpoints_drive_dir": "/content/drive/MyDrive/Projects/Aura/Layer1_SkinTone/checkpoints"
}


## Task Summary — L1_T3 Model Training (Layer 1: Skin Tone / Undertone)

### What was built
| Output | Description | Size | Status |
|---|---|---|---|
| `skin_tone_classifier.tflite` | EfficientNet-B0, 6-class Fitzpatrick scale (I–VI) | 4.55 MB | **Ready for app integration** (see accuracy caveats) |
| `undertone_classifier_WEAK.tflite` | EfficientNet-B0, 3-class undertone (cool / neutral / warm) | 4.55 MB | **Prototype only. Not production-ready.** |

Both models take a `224×224×3` float32 image (raw RGB, 0–255; EfficientNet rescaling is built into the model) and output softmax probabilities.

### Training pipeline
| Stage | Data | What was trained | Result |
|---|---|---|---|
| A | FairFace (race proxy task, ~98k images) | Full backbone adapted from ImageNet toward face/skin features | Proxy task only; accuracy not recorded in the manifest (placeholder `0.0`) |
| B | Fitzpatrick17k (frozen backbone) | New 6-class Fitzpatrick head | Val accuracy **0.360** |
| C | Fitzpatrick17k (top ~20% of backbone unfrozen, LR 1e-5) | Fine-tuning | Val accuracy **0.367** (exact), **0.813** (within ±1 class) |
| D | Roboflow Undertone dataset (980 images; 138 cool / 429 neutral / 413 warm) | New 3-class undertone head on the frozen Stage C backbone | Val accuracy **0.602** |

### Key results and honest limits
- **Fitzpatrick model:** 36.7% exact accuracy on 6 classes (chance ≈ 17%; always predicting the largest class ≈ 29%). Stage C gave no meaningful gain over Stage B. Most errors land in the adjacent class, so the model is useful as a coarse estimator (**81.3% within ±1 class**) but not as a precise classifier. The rare classes (Fitzpatrick I: 24 and VI: 30 validation samples) have low precision (0.20 and 0.32) and unstable results.
- **Undertone model:** the learned head (**59.2%**) beats the rule-based LAB heuristic (**35.7%**, roughly chance), and is in line with the public dataset's reference of about 63%. The dataset comes from a single anonymous uploader, with no published labeling method and no stated demographics, so this is a placeholder, not a validated model.
- **Likely causes of the modest Fitzpatrick accuracy:** noisy crowd-sourced labels in Fitzpatrick17k, a short proxy-task adaptation (3 epochs in Stage A), and images that are often lesion close-ups with uncontrolled lighting rather than face or skin patches.

### Technical notes
- Stage B/C use a **single-output model** (Fitzpatrick only). The dual-output setup failed under Keras 3 and was never needed, since the undertone head was never jointly trained.
- Class imbalance is handled with **per-sample weights** in the training dataset (`class_weight=` does not work reliably for this setup in Keras 3).
- Checkpoints use the `.weights.h5` format. Loading Stage A weights into the new backbone is done by rebuilding the Stage A architecture around it, not with `by_name=True`.
- All checkpoints and both TFLite files are saved and size-verified on Drive after a remount (Step 10). The manifest at `Layer1_SkinTone/manifest.json` is updated.

### Next steps
1. **Replace the proxy data with the in-house, consented, dermatologist-labeled Filipino dataset.** This is the biggest single improvement for both models, and the only way to get a trustworthy undertone model.
2. **Improve the Fitzpatrick model:** train Stage A longer (8–10 epochs), use a higher fine-tuning LR (3e-5 to 1e-4) with more of the backbone unfrozen, add brightness and contrast augmentation, and consider merging to 3 groups (I–II, III–IV, V–VI) if the app doesn't need all six levels.
3. **Evaluate on images that match the real app input** (face or skin photos from the app's camera), since Fitzpatrick17k is only a proxy.
4. **Integrate** `skin_tone_classifier.tflite` into the React Native app, and show Fitzpatrick output as an estimate (for example "Type III–IV"). Do not show the undertone output to users until it has been validated on real data.
5. **Fix the manifest:** `stage_a_race_proxy_val_accuracy` currently holds a placeholder `0.0` from the runtime restart. Replace it with the real Stage A value, or remove the field.
6. **Data credit:** the Roboflow dataset is CC BY 4.0 and needs attribution if the model is distributed.